In [1]:
import csv

In [2]:
accounts = {
    "ac1": "062706",
    "ac2": "081705",
    "ac3": "011114"
}

In [4]:
users = [
    {"acc_no": "ac1", "uname": "Elise", "bal": 50000.0, "logs": []},
    {"acc_no": "ac2", "uname": "Kirsten", "bal": 10000.0, "logs": []},
    {"acc_no": "ac3", "uname": "Alexi", "bal": 15000.0, "logs": []}
]

In [5]:
current_user = None

In [6]:
admin_uname = "admin"
admin_pw = "password"

In [7]:
def client_load():
    client_list = []
    try:
        with open("clients.txt", "r") as f:
            reader = csv.reader(f)
            for line in reader:
                client_list.append({
                    "acc_no": line[0],
                    "uname": line[1],
                    "bal": float(line[2]),
                    "logs": line[3].split(";") if len(line) > 3 else []
                })
    except FileNotFoundError:
        print("Data file not found. Using default clients.")
    return client_list or users

In [8]:
def client_save(client_list):
    with open("clients.txt", "w", newline="") as file:
        writer = csv.writer(file)
        for client in client_list:
            writer.writerow([
                client["acc_no"],
                client["uname"],
                client["bal"],
                ";".join(client["logs"])
            ])
    print("All data saved.")

In [9]:
def login():
    global current_user
    print("Log in to your account")
    ac_id = input("Account: ").strip()
    pin = input("PIN: ").strip()

    for acc, pins in accounts.items():
        if ac_id == acc and pin == pins:
            for usr in users:
                if usr["acc_no"] == ac_id:
                    current_user = usr
                    print(f"Hello, {current_user['uname']}!")
                    return True

    print("Failed login attempt")
    return False

In [11]:
def show_balance():
    print(f"Your current balance is: ${current_user['bal']:.2f}")

In [12]:
def put_money():
    try:
        amount = float(input("Amount to deposit: "))
        if amount < 0:
            print("Negative values not accepted.")
            return

        current_user["bal"] += amount
        current_user["logs"].append(f"Deposited {amount}")
        print("Deposit successful.")

    except ValueError:
        print("Please enter a valid number.")

In [13]:
def take_out_money():
    try:
        amt = float(input("Enter withdraw amount: "))

        if amt < 0:
            print("Cannot withdraw negatives.")
        elif amt > current_user["bal"]:
            print("Insufficient funds.")
        else:
            current_user["bal"] -= amt
            current_user["logs"].append(f"Withdrew {amt}")
            print("Withdrawal successful.")

    except ValueError:
        print("Invalid input!")

In [14]:
def give_money():
    trgt_acc = input("Transfer to: ").strip()

    try:
        tr_amt = float(input("Amount: "))

        if tr_amt <= 0:
            print("Must be a positive value.")
            return

        if tr_amt > current_user["bal"]:
            print("Insufficient funds.")
        else:
            for client in users:
                if client["acc_no"] == trgt_acc:
                    current_user["bal"] -= tr_amt
                    client["bal"] += tr_amt

                    current_user["logs"].append(
                        f"Transferred {tr_amt} to {trgt_acc}"
                    )
                    client["logs"].append(
                        f"Received {tr_amt} from {current_user['acc_no']}"
                    )

                    print("Money sent!")
                    return

            print("Account not found.")

    except ValueError:
        print("Invalid input for transfer.")

In [15]:
def admin_login():
    uname = input("Admin User: ").strip()
    pw = input("Admin Pass: ").strip()

    if uname == admin_uname and pw == admin_pw:
        print("Admin access granted.")
        return True

    print("Denied.")
    return False

In [16]:
def manage():
    while True:
        print("\n-- Admin Menu --")
        print("1. Search Client")
        print("2. Register New User")
        print("3. Remove Client")
        print("4. Exit")

        opt = input("Option: ").strip()

        if opt == "1":
            srch = input("Enter ID or Name: ").lower()
            found = False

            for u in users:
                if u["acc_no"] == srch or u["uname"].lower() == srch:
                    print(f"Found -> {u['uname']}: {u['acc_no']}, ${u['bal']}")
                    found = True
                    break

            if not found:
                print("No matching user.")

        elif opt == "2":
            new_acc = input("Account No.: ").strip()
            new_user = input("Name: ").strip()
            new_pw = input("PIN: ").strip()

            try:
                start_bal = float(input("Starting balance: "))

                accounts[new_acc] = new_pw
                users.append({
                    "acc_no": new_acc,
                    "uname": new_user,
                    "bal": start_bal,
                    "logs": []
                })

                print(f"New user {new_user} registered.")

            except ValueError:
                print("Invalid starting balance.")

        elif opt == "3":
            to_del = input("Account to delete: ").strip()

            for u in users:
                if u["acc_no"] == to_del:
                    users.remove(u)
                    accounts.pop(to_del, None)
                    print(f"User {to_del} removed.")
                    break
            else:
                print("Not found.")

        elif opt == "4":
            print("Back to Main Menu.")
            break

        else:
            print("Invalid choice.")

In [18]:
def main_system():
    global users
    users = client_load()

    while True:
        print("\n1. User Login")
        print("2. Admin Login")
        print("3. Exit")

        c = input("Your option: ").strip()

        if c == "1":
            if login():
                while True:
                    print("\nUser Menu")
                    print("1. Check Balance")
                    print("2. Deposit")
                    print("3. Withdraw")
                    print("4. Transfer")
                    print("5. Logout")

                    ch = input("Choice: ").strip()

                    if ch == "1":
                        show_balance()
                    elif ch == "2":
                        put_money()
                    elif ch == "3":
                        take_out_money()
                    elif ch == "4":
                        give_money()
                    elif ch == "5":
                        print("Logging out...")
                        break
                    else:
                        print("Invalid.")

            else:
                print("Invalid credentials.")

        elif c == "2":
            if admin_login():
                manage()

        elif c == "3":
            client_save(users)
            print("Goodbye!")
            break

        else:
            print("Wrong option.")

In [ ]:
main_system()

Data file not found. Using default clients.

1. User Login
2. Admin Login
3. Exit
Your option: 2
Admin User: admin
Admin Pass: password
Admin access granted.

-- Admin Menu --
1. Search Client
2. Register New User
3. Remove Client
4. Exit
